# Prag-Dristi: Assam Flood Forecasting Engine
### LSTM Encoder-Decoder with Bahdanau Attention

**Author:** Priyanshu Debnath  
**Institution:** Indian Institute of Technology Guwahati  
**Task:** 7-day ahead river discharge forecasting for the Brahmaputra Basin  
**Model:** LSTM Encoder-Decoder with Bahdanau (Additive) Attention  
**Data:** ERA5 Reanalysis + GloFAS-ERA5 River Discharge (Copernicus CDS / EWDS)

---

### Architecture Overview
```
Input Sequence (30 days × 26 features)
        │
   ┌────▼────┐
   │ Encoder │  2-layer stacked LSTM
   │  LSTM   │  hidden_size = 128
   └────┬────┘
        │ all encoder hidden states H = {h₁, ..., h₃₀}
   ┌────▼────────────────┐
   │  Bahdanau Attention │  αₜ,ₛ = softmax(vᵀ tanh(W₁hₛ + W₂hₜ))
   └────┬────────────────┘
        │ context vector cₜ
   ┌────▼────┐
   │ Decoder │  2-layer LSTM, autoregressive
   │  LSTM   │  input = [prev_pred, cₜ]
   └────┬────┘
        │
   ┌────▼────┐
   │  FC Head│  Linear → ReLU → Dropout → Linear
   └────┬────┘
        │
Output: 7-day discharge forecast (log-normalised)
```

### Key References
- Kratzert et al. (2018) — LSTM for rainfall-runoff. *HESS* doi:10.5194/hess-22-6005-2018
- Kao et al. (2020) — LSTM Encoder-Decoder for multi-step flood forecasting. *J. Hydrology* doi:10.1016/j.jhydrol.2020.124631
- Bahdanau et al. (2015) — Additive attention mechanism. *ICLR 2015* arXiv:1409.0473
- Hersbach et al. (2020) — ERA5 global reanalysis. *QJRMS* doi:10.1002/qj.3803
- Harrigan et al. (2020) — GloFAS-ERA5 river discharge reanalysis. *ESSD* doi:10.5194/essd-12-2043-2020

## 0. Environment Setup

In [ ]:
import os, sys
import json
from pathlib import Path

# ── Detect environment ──────────────────────────────────────────────────────
ON_KAGGLE = Path('/kaggle').exists()
ON_COLAB  = 'google.colab' in sys.modules

if ON_KAGGLE:
    ROOT = Path('/kaggle/working/prag-dristi')
    DATA_DIR = Path('/kaggle/input/datasets/priyanshudebnathiitg/assam-flood-forecasting-era5-glofas')   # your uploaded dataset
    print('Running on Kaggle')
elif ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = Path('/content/prag-dristi')
    DATA_DIR = Path('/content/drive/MyDrive/prag-dristi/data')
    print('Running on Colab')
else:
    ROOT = Path('.').resolve().parent
    DATA_DIR = ROOT / 'data'
    print(f'Running locally: {ROOT}')

sys.path.insert(0, str(ROOT))
print(f'ROOT      : {ROOT}')
print(f'DATA_DIR  : {DATA_DIR}')

In [ ]:
# ── Install any missing dependencies ────────────────────────────────────────
if ON_KAGGLE or ON_COLAB:
    os.system('pip install -q hydra-core omegaconf joblib')

import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
import joblib
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch   : {torch.__version__}')
print(f'CUDA      : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU       : {torch.cuda.get_device_name(0)}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device    : {DEVICE}')

## 1. Configuration

In [ ]:
# ── All hyperparameters in one place ────────────────────────────────────────
CFG = {
    # Data
    'station'         : 'bahadurabad',
    'encoder_len'     : 30,       # days of history fed to encoder
    'decoder_len'     : 7,        # days ahead to forecast
    'train_frac'      : 0.70,
    'val_frac'        : 0.15,
    # Model
    'hidden_size'     : 128,
    'num_layers'      : 2,
    'dropout'         : 0.3,
    'fc_hidden'       : 64,
    'use_attention'   : True,
    # Training
    'batch_size'      : 64,
    'epochs'          : 100,
    'lr'              : 1e-3,
    'weight_decay'    : 1e-5,
    'grad_clip'       : 1.0,
    'patience'        : 15,
    'flood_threshold' : 98600.0,  # m³/s danger level at Bahadurabad
    'flood_weight'    : 4.0,      # loss multiplier for flood events
    # Misc
    'seed'            : 42,
    'log_every'       : 5,
}

CKPT_DIR = ROOT / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG['seed'])
print('Configuration set.')

## 2. Data Loading and Feature Engineering

Features engineered from ERA5 + GloFAS-ERA5:

| Category | Features | Motivation |
|---|---|---|
| Meteorological | precip_mm, temp_c, soil_moisture, wind_speed | Primary flood drivers (ERA5) |
| Discharge lags | Q(t-1) ... Q(t-7) | Persistence of river flow |
| Rainfall lags | P(t-1) ... P(t-7) | Upstream routing lag (~3-7 days) |
| Rolling rainfall | 3d, 7d, 14d, 30d sums | Soil saturation accumulation |
| Rolling discharge | 3d, 7d means | Baseflow trend |
| Log discharge | log(1+Q) | Regime indicator (flood vs. normal) |
| Seasonality | sin(2πd/365), cos(2πd/365) | Monsoon cycle encoding |

In [ ]:
# ── Load merged CSV ──────────────────────────────────────────────────────────
csv_path = DATA_DIR / f"merged_{CFG['station']}.csv"
assert csv_path.exists(), f"File not found: {csv_path}\nRun build_dataset.py first."

df_raw = pd.read_csv(csv_path, index_col='date', parse_dates=True)
print(f'Loaded {len(df_raw):,} rows  |  {df_raw.index[0].date()} → {df_raw.index[-1].date()}')
print(f'Columns: {list(df_raw.columns)}')
df_raw.head()

In [ ]:
# ── Feature engineering ──────────────────────────────────────────────────────
def engineer_features(df):
    df = df.copy().sort_index()
    DISCHARGE_COL = 'discharge_m3s'

    # Wind speed magnitude
    if 'wind_u' in df.columns and 'wind_v' in df.columns:
        df['wind_speed'] = np.sqrt(df['wind_u']**2 + df['wind_v']**2)
        df.drop(columns=['wind_u', 'wind_v'], inplace=True)

    # Lag features
    for lag in [1, 2, 3, 5, 7]:
        df[f'discharge_lag{lag}'] = df[DISCHARGE_COL].shift(lag)
        if 'precip_mm' in df.columns:
            df[f'precip_lag{lag}'] = df['precip_mm'].shift(lag)

    # Rolling rainfall accumulations
    if 'precip_mm' in df.columns:
        for w in [3, 7, 14, 30]:
            df[f'precip_roll{w}d'] = df['precip_mm'].rolling(w, min_periods=1).sum()

    # Rolling discharge statistics
    for w in [3, 7]:
        df[f'discharge_roll{w}d_mean'] = df[DISCHARGE_COL].rolling(w, min_periods=1).mean()

    # Log discharge (regime indicator)
    df['log_discharge'] = np.log1p(df[DISCHARGE_COL])

    # Sinusoidal seasonality encoding
    doy = df.index.dayofyear
    df['doy_sin'] = np.sin(2 * np.pi * doy / 365.25)
    df['doy_cos'] = np.cos(2 * np.pi * doy / 365.25)

    df.dropna(inplace=True)
    return df

df = engineer_features(df_raw)
print(f'After feature engineering: {len(df):,} rows, {len(df.columns)} columns')

FEATURE_COLS = [c for c in df.columns if c not in ['discharge_m3s', 'log_discharge']]
print(f'Feature columns ({len(FEATURE_COLS)}): {FEATURE_COLS}')

In [ ]:
# ── Temporal train/val/test split (NEVER shuffle) ───────────────────────────
n = len(df)
n_train = int(n * CFG['train_frac'])
n_val   = int(n * CFG['val_frac'])

train_df = df.iloc[:n_train]
val_df   = df.iloc[n_train : n_train + n_val]
test_df  = df.iloc[n_train + n_val:]

print(f'Train : {len(train_df):>5,} rows  {train_df.index[0].date()} → {train_df.index[-1].date()}')
print(f'Val   : {len(val_df):>5,} rows  {val_df.index[0].date()}  → {val_df.index[-1].date()}')
print(f'Test  : {len(test_df):>5,} rows  {test_df.index[0].date()}  → {test_df.index[-1].date()}')

# Fit scalers on training data ONLY
feat_scaler = StandardScaler()
feat_scaler.fit(train_df[FEATURE_COLS].values)

tgt_scaler = StandardScaler()
tgt_scaler.fit(train_df[['log_discharge']].values)

joblib.dump({'feature': feat_scaler, 'target': tgt_scaler}, CKPT_DIR / 'scalers.pkl')
print('Scalers fitted and saved.')

In [ ]:
# ── PyTorch Dataset ──────────────────────────────────────────────────────────
class FloodDataset(torch.utils.data.Dataset):
    def __init__(self, df, feature_cols, feat_scaler, tgt_scaler, enc_len, dec_len):
        feats  = feat_scaler.transform(df[feature_cols].values).astype(np.float32)
        tgts   = tgt_scaler.transform(df[['log_discharge']].values).astype(np.float32).squeeze()
        raw_tg = df['discharge_m3s'].values.astype(np.float32)

        self.feats, self.tgts, self.raw_tg = feats, tgts, raw_tg
        self.enc_len, self.dec_len = enc_len, dec_len
        self.n = len(df) - enc_len - dec_len + 1

    def __len__(self):  return self.n

    def __getitem__(self, i):
        e = i + self.enc_len
        return {
            'x'    : torch.from_numpy(self.feats[i:e]),
            'y'    : torch.from_numpy(self.tgts[e : e + self.dec_len]),
            'y_raw': torch.from_numpy(self.raw_tg[e : e + self.dec_len]),
        }

EL, DL = CFG['encoder_len'], CFG['decoder_len']
ds_kw  = dict(feature_cols=FEATURE_COLS, feat_scaler=feat_scaler,
               tgt_scaler=tgt_scaler, enc_len=EL, dec_len=DL)

train_loader = DataLoader(FloodDataset(train_df, **ds_kw),
                          batch_size=CFG['batch_size'], shuffle=True,  pin_memory=True)
val_loader   = DataLoader(FloodDataset(val_df,   **ds_kw),
                          batch_size=CFG['batch_size'], shuffle=False)
test_loader  = DataLoader(FloodDataset(test_df,  **ds_kw),
                          batch_size=CFG['batch_size'], shuffle=False)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Test  batches : {len(test_loader)}')

## 3. Model Architecture

**Bahdanau (Additive) Attention** (Bahdanau et al., 2015):
$$e_{t,s} = v^\top \tanh(W_1 h_s^{enc} + W_2 h_t^{dec})$$
$$\alpha_{t,s} = \frac{\exp(e_{t,s})}{\sum_{s'} \exp(e_{t,s'})}$$
$$c_t = \sum_s \alpha_{t,s} \cdot h_s^{enc}$$

In [ ]:
class BahdanauAttention(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.W_enc = nn.Linear(hidden_size, hidden_size, bias=False)
        self.W_dec = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v     = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, dec_h, enc_out):
        # dec_h  : (B, H)          enc_out : (B, L, H)
        energy  = torch.tanh(self.W_enc(enc_out) + self.W_dec(dec_h.unsqueeze(1)))
        weights = torch.softmax(self.v(energy).squeeze(-1), dim=-1)   # (B, L)
        context = torch.bmm(weights.unsqueeze(1), enc_out).squeeze(1) # (B, H)
        return context, weights


class FloodForecastModel(nn.Module):
    """
    LSTM Encoder-Decoder with Bahdanau Attention for multi-step
    ahead river discharge forecasting.

    Reference: Kao et al. (2020) J. Hydrology doi:10.1016/j.jhydrol.2020.124631
    """
    def __init__(self, input_size, hidden_size, num_layers, dropout,
                 decoder_len, fc_hidden, use_attention=True):
        super().__init__()
        self.decoder_len  = decoder_len
        self.use_attention = use_attention

        self.encoder = nn.LSTM(input_size, hidden_size, num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0.0)

        dec_input_size = 1 + (hidden_size if use_attention else 0)
        self.decoder = nn.LSTM(dec_input_size, hidden_size, num_layers,
                               batch_first=True,
                               dropout=dropout if num_layers > 1 else 0.0)

        if use_attention:
            self.attention = BahdanauAttention(hidden_size)

        self.fc = nn.Sequential(
            nn.Linear(hidden_size, fc_hidden), nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fc_hidden, 1)
        )

    def forward(self, x, teacher_forcing_ratio=0.0, targets=None):
        enc_out, (h, c) = self.encoder(x)         # enc_out: (B, L, H)
        B = x.size(0)
        prev = torch.zeros(B, 1, device=x.device)
        preds = []

        for t in range(self.decoder_len):
            if self.use_attention:
                ctx, _ = self.attention(h[-1], enc_out)
                dec_in = torch.cat([prev, ctx], dim=-1).unsqueeze(1)
            else:
                dec_in = prev.unsqueeze(1)

            out, (h, c) = self.decoder(dec_in, (h, c))
            pred = self.fc(out.squeeze(1))         # (B, 1)
            preds.append(pred)

            use_tf = (teacher_forcing_ratio > 0 and
                      targets is not None and
                      torch.rand(1).item() < teacher_forcing_ratio)
            prev = targets[:, t].unsqueeze(1) if use_tf else pred

        return torch.cat(preds, dim=1)  # (B, decoder_len)


model = FloodForecastModel(
    input_size   = len(FEATURE_COLS),
    hidden_size  = CFG['hidden_size'],
    num_layers   = CFG['num_layers'],
    dropout      = CFG['dropout'],
    decoder_len  = CFG['decoder_len'],
    fc_hidden    = CFG['fc_hidden'],
    use_attention= CFG['use_attention'],
).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters : {n_params:,}')
print(model)

## 4. Training

**Flood-Weighted Loss** — standard MSE treats all days equally, causing the model
to optimise for the 92%+ of non-flood days at the expense of rare flood peaks.
We apply a weight proportional to discharge magnitude:

$$w_i = 1 + \lambda \cdot \text{clip}\left(\frac{Q_i}{Q_{danger}}, 0, 1\right), \quad \lambda=4$$

$$\mathcal{L} = \frac{1}{N} \sum_i w_i (\hat{Q}'_i - Q'_i)^2$$

Normal days → weight 1.0 &nbsp;|&nbsp; Flood-peak days → weight 5.0

In [ ]:
def flood_weighted_loss(preds, targets, tgt_scaler, flood_threshold, lam):
    """Flood-aware MSE: errors on high-discharge days penalised more."""
    targets_raw = torch.tensor(
        np.expm1(tgt_scaler.inverse_transform(
            targets.detach().cpu().numpy().reshape(-1, 1)
        ).reshape(targets.shape)),
        device=targets.device
    )
    weights = 1.0 + lam * torch.clamp(targets_raw / flood_threshold, 0.0, 1.0)
    return (weights * (preds - targets) ** 2).mean()


@torch.no_grad()
def compute_metrics(obs, sim, threshold):
    """NSE, KGE, RMSE, and flood verification metrics."""
    obs, sim = np.asarray(obs, float), np.asarray(sim, float)
    mask = np.isfinite(obs) & np.isfinite(sim)
    obs, sim = obs[mask], sim[mask]

    # NSE
    nse = 1 - np.sum((obs-sim)**2) / (np.sum((obs-np.mean(obs))**2) + 1e-9)
    # KGE
    r = np.corrcoef(obs, sim)[0,1]
    a = np.std(sim)  / (np.std(obs)  + 1e-9)
    b = np.mean(sim) / (np.mean(obs) + 1e-9)
    kge = 1 - np.sqrt((r-1)**2 + (a-1)**2 + (b-1)**2)
    # RMSE
    rmse = np.sqrt(np.mean((obs-sim)**2))
    # Flood contingency
    of, sf = obs >= threshold, sim >= threshold
    TP = np.sum(of & sf); FP = np.sum(~of & sf)
    FN = np.sum(of & ~sf)
    csi = TP / (TP+FP+FN+1e-9)
    pod = TP / (TP+FN+1e-9)
    far = FP / (TP+FP+1e-9)
    return dict(NSE=nse, KGE=kge, RMSE=rmse, CSI=csi, POD=pod, FAR=far)


def evaluate(model, loader, tgt_scaler, flood_threshold, lam):
    model.eval()
    total_loss, all_obs, all_sim = 0.0, [], []
    with torch.no_grad():
        for batch in loader:
            x, y, y_raw = batch['x'].to(DEVICE), batch['y'].to(DEVICE), batch['y_raw'].numpy()
            preds = model(x)
            total_loss += flood_weighted_loss(preds, y, tgt_scaler, flood_threshold, lam).item() * x.size(0)
            preds_np  = preds.cpu().numpy()
            preds_inv = tgt_scaler.inverse_transform(preds_np.reshape(-1,1)).reshape(preds_np.shape)
            preds_m3s = np.expm1(preds_inv)
            all_obs.append(y_raw); all_sim.append(preds_m3s)

    obs = np.concatenate(all_obs).ravel()
    sim = np.concatenate(all_sim).ravel()
    metrics = compute_metrics(obs, sim, flood_threshold)
    metrics['loss'] = total_loss / len(loader.dataset)
    return metrics

print('Loss and evaluation functions defined.')

In [ ]:
# ── Training loop ────────────────────────────────────────────────────────────
optimizer = torch.optim.Adam(model.parameters(),
                             lr=CFG['lr'], weight_decay=CFG['weight_decay'])
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, patience=5, factor=0.5)

best_val_loss   = float('inf')
patience_ctr    = 0
best_ckpt       = CKPT_DIR / 'best_model.pt'
history         = {'train_loss': [], 'val_loss': [], 'val_nse': [], 'val_kge': []}

print(f'Starting training for up to {CFG["epochs"]} epochs ...')
print(f'{"Epoch":>6}  {"Train Loss":>12}  {"Val Loss":>10}  {"Val NSE":>9}  {"Val KGE":>9}')
print('-' * 58)

for epoch in range(1, CFG['epochs'] + 1):
    # ── teacher forcing anneals from 0.5 → 0 over first 30 epochs ──────────
    tf = max(0.0, 0.5 * (1.0 - epoch / 30))

    # ── train ────────────────────────────────────────────────────────────────
    model.train()
    epoch_loss = 0.0
    for batch in train_loader:
        x, y = batch['x'].to(DEVICE), batch['y'].to(DEVICE)
        optimizer.zero_grad()
        preds = model(x, teacher_forcing_ratio=tf, targets=y)
        loss  = flood_weighted_loss(preds, y, tgt_scaler,
                                    CFG['flood_threshold'], CFG['flood_weight'])
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        optimizer.step()
        epoch_loss += loss.item() * x.size(0)
    epoch_loss /= len(train_loader.dataset)

    # ── validate ─────────────────────────────────────────────────────────────
    val_metrics = evaluate(model, val_loader, tgt_scaler,
                           CFG['flood_threshold'], CFG['flood_weight'])
    val_loss = val_metrics['loss']
    scheduler.step(val_loss)

    history['train_loss'].append(epoch_loss)
    history['val_loss'].append(val_loss)
    history['val_nse'].append(val_metrics['NSE'])
    history['val_kge'].append(val_metrics['KGE'])

    if epoch % CFG['log_every'] == 0 or epoch == 1:
        print(f'{epoch:>6}  {epoch_loss:>12.5f}  {val_loss:>10.5f}  '
              f'{val_metrics["NSE"]:>9.4f}  {val_metrics["KGE"]:>9.4f}')

    # ── early stopping + checkpoint ──────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), best_ckpt)
        patience_ctr = 0
    else:
        patience_ctr += 1
        if patience_ctr >= CFG['patience']:
            print(f'\nEarly stopping at epoch {epoch}.')
            break

print(f'\nBest val loss : {best_val_loss:.5f}')
print(f'Checkpoint    : {best_ckpt}')

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
epochs_ran = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_ran, history['train_loss'], label='Train', color='#1f77b4')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val',   color='#ff7f0e')
axes[0].set_title('Flood-Weighted Loss'); axes[0].set_xlabel('Epoch')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history['val_nse'], color='#2ca02c')
axes[1].axhline(0.65, ls='--', color='gray', label='Good threshold (0.65)')
axes[1].set_title('Validation NSE'); axes[1].set_xlabel('Epoch')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(epochs_ran, history['val_kge'], color='#d62728')
axes[2].axhline(0.60, ls='--', color='gray', label='Good threshold (0.60)')
axes[2].set_title('Validation KGE'); axes[2].set_xlabel('Epoch')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Prag-Dristi Training Curves — LSTM Encoder-Decoder + Attention',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(CKPT_DIR / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved training_curves.png')

## 6. Test Set Evaluation

Performance thresholds (Nash & Sutcliffe, 1970; Gupta et al., 2009):

| NSE | KGE | Rating |
|---|---|---|
| > 0.75 | > 0.70 | Very Good |
| 0.65–0.75 | 0.60–0.70 | Good |
| 0.50–0.65 | 0.40–0.60 | Satisfactory |
| < 0.50 | < 0.40 | Unsatisfactory |

In [ ]:
# ── Load best checkpoint ─────────────────────────────────────────────────────
model.load_state_dict(torch.load(best_ckpt, map_location=DEVICE))
test_metrics = evaluate(model, test_loader, tgt_scaler,
                        CFG['flood_threshold'], CFG['flood_weight'])

print('=' * 45)
print('  TEST SET RESULTS — Prag-Dristi v1.0')
print('=' * 45)
for k, v in test_metrics.items():
    bar = '█' * int(max(0, v) * 20) if k in ('NSE','KGE','CSI','POD') else ''
    print(f'  {k:<10} {v:>10.4f}  {bar}')
print('=' * 45)

with open(CKPT_DIR / 'results.json', 'w') as f:
    json.dump({k: round(float(v), 4) for k, v in test_metrics.items()}, f, indent=2)
print('Results saved to checkpoints/results.json')

## 7. Forecast Visualisation

In [ ]:
# ── Collect all test predictions ─────────────────────────────────────────────
model.eval()
all_obs_full, all_sim_full = [], []

with torch.no_grad():
    for batch in test_loader:
        x = batch['x'].to(DEVICE)
        y_raw = batch['y_raw'].numpy()
        preds_np  = model(x).cpu().numpy()
        preds_inv = tgt_scaler.inverse_transform(preds_np.reshape(-1,1)).reshape(preds_np.shape)
        preds_m3s = np.expm1(preds_inv)
        all_obs_full.append(y_raw)
        all_sim_full.append(preds_m3s)

obs_all = np.concatenate(all_obs_full).ravel()
sim_all = np.concatenate(all_sim_full).ravel()

# ── Figure 1: Observed vs Predicted time series ──────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 9))

n_plot = min(730, len(obs_all))  # plot 2 years max
axes[0].plot(obs_all[:n_plot], label='Observed',  color='#1f77b4', lw=1.2, alpha=0.9)
axes[0].plot(sim_all[:n_plot], label='Predicted', color='#ff7f0e', lw=1.0, alpha=0.85, ls='--')
axes[0].axhline(CFG['flood_threshold'], color='red', ls=':', lw=1.5,
                label=f'Danger level ({CFG["flood_threshold"]/1e3:.0f}k m³/s)')
axes[0].set_title('Brahmaputra Discharge — Observed vs Predicted (Test Set)',
                  fontsize=13, fontweight='bold')
axes[0].set_ylabel('Discharge (m³/s)'); axes[0].legend(); axes[0].grid(alpha=0.3)

# ── Scatter: observed vs predicted ───────────────────────────────────────────
axes[1].scatter(obs_all[::3], sim_all[::3], alpha=0.25, s=8, color='#1f77b4')
lim = max(obs_all.max(), sim_all.max()) * 1.05
axes[1].plot([0, lim], [0, lim], 'r--', lw=1.5, label='1:1 line')
axes[1].set_xlabel('Observed (m³/s)'); axes[1].set_ylabel('Predicted (m³/s)')
axes[1].set_title(f'Scatter — NSE={test_metrics["NSE"]:.3f}, KGE={test_metrics["KGE"]:.3f}',
                  fontsize=12)
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CKPT_DIR / 'forecast_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved forecast_plot.png')

In [ ]:
# ── Figure 2: Flood event zoom ────────────────────────────────────────────────
# Find the monsoon window with the highest observed discharge
peak_idx = int(np.argmax(obs_all))
start    = max(0, peak_idx - 60)
end      = min(len(obs_all), peak_idx + 60)

fig, ax = plt.subplots(figsize=(14, 5))
ax.fill_between(range(end-start), obs_all[start:end], alpha=0.25, color='#1f77b4')
ax.plot(obs_all[start:end], label='Observed',  color='#1f77b4', lw=1.8)
ax.plot(sim_all[start:end], label='Predicted', color='#ff7f0e', lw=1.6, ls='--')
ax.axhline(CFG['flood_threshold'], color='red', ls=':', lw=1.5, label='Danger level')
ax.set_title('Flood Event Zoom — Peak Discharge Period (Test Set)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Days from window start'); ax.set_ylabel('Discharge (m³/s)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CKPT_DIR / 'flood_zoom.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Summary

In [ ]:
print()
print('╔══════════════════════════════════════════════════════╗')
print('║         PRAG-DRISTI v1.0 — TRAINING COMPLETE        ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Station        : Bahadurabad (Brahmaputra)          ║')
print(f'║  Architecture   : LSTM Enc-Dec + Bahdanau Attention  ║')
print(f'║  Parameters     : {n_params:,}                         ║')
print(f'║  Forecast lead  : 7 days                             ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  NSE            : {test_metrics["NSE"]:>6.4f}                            ║')
print(f'║  KGE            : {test_metrics["KGE"]:>6.4f}                            ║')
print(f'║  RMSE (m³/s)    : {test_metrics["RMSE"]:>10.1f}                        ║')
print(f'║  CSI (floods)   : {test_metrics["CSI"]:>6.4f}                            ║')
print(f'║  POD            : {test_metrics["POD"]:>6.4f}                            ║')
print(f'║  FAR            : {test_metrics["FAR"]:>6.4f}                            ║')
print('╠══════════════════════════════════════════════════════╣')
print(f'║  Checkpoint     : checkpoints/best_model.pt          ║')
print(f'║  Scalers        : checkpoints/scalers.pkl            ║')
print(f'║  Results        : checkpoints/results.json           ║')
print('╚══════════════════════════════════════════════════════╝')

# Download artefacts on Kaggle/Colab
if ON_KAGGLE:
    import shutil
    shutil.make_archive('/kaggle/working/prag_dristi_checkpoints', 'zip', str(CKPT_DIR))
    print('Zipped checkpoints -> /kaggle/working/prag_dristi_checkpoints.zip')